# Phase E — EXPLORE
## Statistical EDA → Data-Driven Hyperparameter Decisions

**OSEMN Step 3**: Understand the data before modelling. The outputs of this notebook
directly inform model design:

| EDA Output | → Model Decision |
|---|---|
| Stationarity (ADF+KPSS) | Which features to include in HMM |
| Correlation pairs | Drop redundant features |
| BIC elbow chart | Number of HMM states K |
| Distribution skewness | Gaussian vs t-distribution for emissions |
| NBER regime overlap | Validate model concept |

> This is where the 'traditional approach with deviations' happens —
> K is NOT hardcoded to 3. BIC tells us the answer.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import matplotlib.pyplot as plt

from src.config_loader import load_config
from src.obtain.fred_loader import FREDLoader
from src.obtain.market_loader import MarketLoader
from src.scrub.preprocessor import Preprocessor
from src.scrub.feature_engineer import FeatureEngineer
from src.explore.eda_stats import EDAStats
from src.explore.eda_plots import EDAPlots
from src.model.hmm_selector import HMMSelector

cfg = load_config('../config/config.yaml')

# Load processed features (run 02_scrub first)
fred_data = FREDLoader(cfg).load()
market_data = MarketLoader(cfg).load()
preprocessor = Preprocessor(cfg)
df_zscored, df_raw = preprocessor.build_feature_matrix(fred_data, save=False)
features = FeatureEngineer(cfg).engineer(df_raw, df_zscored, save=False)
print(f'Features: {features.shape}')

## 3.1 Stationarity Tests (ADF + KPSS)

HMM assumes stationary emissions. Non-stationary features need differencing.

In [ ]:
eda = EDAStats(cfg)
stat_report = eda.stationarity_report(features)

print('\n=== STATIONARITY REPORT ===')
print(stat_report.to_string(index=False))

# Data-driven: use stationary features for HMM
recommended_features = eda.recommended_features_for_hmm(stat_report)
print(f'\nRecommended HMM features ({len(recommended_features)}): {recommended_features}')
print('\nUpdate config[model][hmm][features_for_hmm] with this list if desired.')

## 3.2 Descriptive Statistics & Normality

In [ ]:
desc = eda.descriptive_stats(features)
print('Descriptive Statistics:')
desc.round(3)

In [ ]:
norm_tests = eda.normality_tests(features)
print('Normality (Jarque-Bera) — non-normal features may need t-distribution emissions:')
print(norm_tests[norm_tests['is_normal']==False].to_string(index=False))

## 3.3 Correlation Heatmap — Identify Redundant Features

In [ ]:
plots = EDAPlots(cfg)
corr = eda.correlation_matrix(features)
plots.correlation_heatmap(corr)

high_corr = eda.high_correlation_pairs(corr)
print(f'\nHighly correlated pairs (|r| > {cfg["exploration"]["high_corr_threshold"]}):')
print(high_corr.to_string(index=False) if not high_corr.empty else 'None')

## 3.4 Feature Distributions

In [ ]:
plots.feature_timeseries(features)
plots.distribution_grid(features)
print('Plots saved to outputs/figures/')

## 3.5 BIC Selection — Data-Driven Choice of K

This is the key modelling decision. We let the data tell us K.

> ⏳ **This cell takes ~5-10 minutes** (50 random restarts × 4 state counts)
> 
> Reduce `n_fits` in config for faster results during exploration.

In [ ]:
oos_start = cfg['exploration']['oos_start']
train_features = features[features.index < oos_start].dropna()
X_train = train_features.values
print(f'Training data for BIC: {X_train.shape} ({train_features.index[0].date()} → {train_features.index[-1].date()})')

selector = HMMSelector(cfg)
sel = selector.select(X_train)

print(f'\nBIC scores: {sel["bic_scores"]}')
print(f'AIC scores: {sel["aic_scores"]}')
print(f'\n→ SELECTED K = {sel["best_k"]} states')

rec = eda.bic_recommendation(sel['bic_scores'])
print(f'\n{rec["reasoning"]}')

In [ ]:
plots.bic_elbow(sel['bic_scores'], sel['aic_scores'])
print('BIC elbow chart saved.')

## 3.6 Decision — Record Findings for Model Phase

In [ ]:
print('=== EDA DECISIONS FOR MODEL PHASE ===')
print(f'Recommended K (states):         {sel["best_k"]}')
print(f'Stationary features to use:     {recommended_features}')
print(f'Non-normal features (use t-dist?): '
      f'{norm_tests[norm_tests["is_normal"]==False]["feature"].tolist()[:5]}')
print()
print('To update config, set:')
print(f'  model.hmm.n_states_range: [{sel["best_k"]}, {sel["best_k"]}]  (lock to selected K)')
print(f'  model.hmm.features_for_hmm: {recommended_features}')

## Summary
- Stationarity: identified which features are valid HMM inputs
- Correlation: flagged redundant pairs (consider dropping)
- BIC: selected optimal K states **from data** (not assumed = 3)
- Distribution: noted fat-tails for potential t-distribution emissions
- **Next**: `04_model.ipynb` — walk-forward HMM training at selected K